# Check acceleration traces for artifacts across all sessions

Load all flights and compute acceleration from position to identify
artifacts like those found in Dataset_3_29959_240403 clusters 27 vs 28.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.io import loadmat
import glob
import os
import sys

sys.path.append(os.path.expanduser('~/Documents/code/digitalbat'))
from digitalbat.utility.utils import FlightClusters

In [ ]:
PARENT_FOLDER = '/mnt/s5/Adam/Angelo/NP_datasets/'
SESSION_NAMES = [
    'Dataset_3_14543_240419',
    'Dataset_3_14543_240420',
    'Dataset_3_14543_240421',
    'Dataset_3_29959_240402',
    'Dataset_3_29959_240403',
    'Dataset_3_29959_240404',
    'Dataset_2_14445_231208',
    'Dataset_2_14445_231209',
    'Dataset_1_32622_231009',
    'Dataset_1_32622_231010',
]
MIN_FLIGHT_DUR = 1.0  # seconds
BIN_WIDTH = 0.10  # seconds, matching model config

## Load raw behavior and cluster flights for all sessions

In [ ]:
all_session_flights = {}

for session_name in SESSION_NAMES:
    print(f"\n{'='*60}\n{session_name}")
    
    # Load raw behavior
    bhv_file = glob.glob(PARENT_FOLDER + session_name + '/Extracted_Behavior_*.mat')[0]
    bhv = loadmat(bhv_file)
    
    r = bhv['r']
    if 'Dataset_2' in session_name:
        t_bhv = bhv['t'][:, 0]
    else:
        t_bhv = bhv['t'][0, :]
    bflying = bhv['bflying'][:, 0].astype(int)
    Fs = float(bhv['Fs'][0, 0])
    
    # Run flight clustering
    fc = FlightClusters(r_vector=r, f_vector=bflying, Fs=Fs,
                        frechet_clus=True, alpha=0.8)
    fc_results = fc.compute()
    
    cluster_ids = fc_results['id']
    f_start = fc_results['start_frame']
    f_stop = fc_results['stop_frame']
    n_flights = fc_results['N']
    
    # Build per-flight DataFrame with acceleration stats
    rows = []
    for i in range(n_flights):
        dur_frames = f_stop[i] - f_start[i] + 1
        if dur_frames / Fs < MIN_FLIGHT_DUR:
            continue
        
        idx = slice(f_start[i], f_stop[i] + 1)
        pos = r[idx, :]  # (T, 3)
        t = t_bhv[idx]
        
        # Velocity and acceleration via finite differences
        vel = np.gradient(pos, 1.0 / Fs, axis=0)  # m/s
        acc = np.gradient(vel, 1.0 / Fs, axis=0)   # m/s^2
        acc_mag = np.linalg.norm(acc, axis=1)
        speed = np.linalg.norm(vel, axis=1)
        
        rows.append(dict(
            session=session_name,
            flight_idx=i,
            cluster=cluster_ids[i],
            duration_s=dur_frames / Fs,
            max_acc=acc_mag.max(),
            mean_acc=acc_mag.mean(),
            p99_acc=np.percentile(acc_mag, 99),
            max_speed=speed.max(),
            mean_speed=speed.mean(),
            pos=pos,
            vel=vel,
            acc=acc,
            acc_mag=acc_mag,
            speed=speed,
            t=t,
        ))
    
    df = pd.DataFrame(rows)
    all_session_flights[session_name] = df
    
    n_clusters = len(df.cluster.unique())
    print(f"  Fs={Fs} Hz, {len(df)} flights (>={MIN_FLIGHT_DUR}s), {n_clusters} clusters")
    print(f"  Max acceleration across all flights: {df.max_acc.max():.1f} m/s²")
    print(f"  Top 5 flights by max acceleration:")
    top5 = df.nlargest(5, 'max_acc')[['flight_idx', 'cluster', 'duration_s', 'max_acc', 'mean_acc']]
    print(top5.to_string(index=False))

## Distribution of max acceleration across all sessions

In [ ]:
# Combine all sessions into one DataFrame (without the array columns)
summary_rows = []
for session_name, df in all_session_flights.items():
    for _, row in df.iterrows():
        summary_rows.append(dict(
            session=row.session,
            flight_idx=row.flight_idx,
            cluster=row.cluster,
            duration_s=row.duration_s,
            max_acc=row.max_acc,
            mean_acc=row.mean_acc,
            p99_acc=row.p99_acc,
            max_speed=row.max_speed,
            mean_speed=row.mean_speed,
        ))
summary_df = pd.DataFrame(summary_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Max acceleration distribution per session
sns.boxplot(data=summary_df, x='session', y='max_acc', ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_ylabel('Max acceleration (m/s²)')
axes[0].set_title('Max acceleration per flight')

# p99 acceleration
sns.boxplot(data=summary_df, x='session', y='p99_acc', ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_ylabel('99th percentile acceleration (m/s²)')
axes[1].set_title('P99 acceleration per flight')

plt.tight_layout()
plt.show()

## Flag outlier flights (potential artifacts)

Identify flights with unusually high acceleration that may indicate tracking artifacts.

In [ ]:
# Flag flights with max acceleration > threshold (gravity ~ 9.8 m/s², bats rarely exceed ~3g)
ACC_THRESHOLD = 50  # m/s², adjust as needed

outliers = summary_df[summary_df.max_acc > ACC_THRESHOLD].sort_values('max_acc', ascending=False)
print(f"Flights with max acceleration > {ACC_THRESHOLD} m/s²: {len(outliers)}")
print()
outliers[['session', 'flight_idx', 'cluster', 'duration_s', 'max_acc', 'mean_acc', 'p99_acc']]

## Acceleration traces for each cluster within each session

Plot mean ± std of acceleration magnitude for each cluster, overlaying individual flights. Artifacts will show as sharp spikes.

In [ ]:
for session_name, df in all_session_flights.items():
    clusters = sorted(df.cluster.unique())
    n_clusters = len(clusters)
    if n_clusters == 0:
        continue
    
    ncols = 6
    nrows = int(np.ceil(n_clusters / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.5), squeeze=False)
    fig.suptitle(f'{session_name}', fontsize=14, y=1.02)
    
    for idx, cid in enumerate(clusters):
        ax = axes[idx // ncols, idx % ncols]
        cluster_flights = df[df.cluster == cid]
        
        max_acc_in_cluster = 0
        for _, row in cluster_flights.iterrows():
            # Normalize time to [0, 1] fractional progress
            acc_mag = row.acc_mag
            t_norm = np.linspace(0, 1, len(acc_mag))
            ax.plot(t_norm, acc_mag, alpha=0.3, linewidth=0.8)
            max_acc_in_cluster = max(max_acc_in_cluster, acc_mag.max())
        
        ax.set_title(f'C{cid} (n={len(cluster_flights)})', fontsize=9)
        ax.set_xlabel('Frac. time', fontsize=7)
        ax.set_ylabel('|acc| (m/s²)', fontsize=7)
        ax.tick_params(labelsize=6)
        
        # Highlight if any flight exceeds threshold
        if max_acc_in_cluster > ACC_THRESHOLD:
            ax.set_facecolor('#fff0f0')
            ax.set_title(f'C{cid} (n={len(cluster_flights)}) ⚠', fontsize=9, color='red')
    
    # Hide unused axes
    for idx in range(n_clusters, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)
    
    plt.tight_layout()
    plt.show()

## Detailed view of flagged clusters

For clusters with outlier acceleration, plot position, velocity, and acceleration traces side by side.

In [ ]:
# Get unique (session, cluster) pairs that have outlier flights
flagged = outliers[['session', 'cluster']].drop_duplicates()

for _, flag_row in flagged.iterrows():
    session_name = flag_row.session
    cid = flag_row.cluster
    df = all_session_flights[session_name]
    cluster_flights = df[df.cluster == cid]
    
    n_flights = len(cluster_flights)
    fig, axes = plt.subplots(n_flights, 4, figsize=(16, 3 * n_flights), squeeze=False)
    fig.suptitle(f'{session_name} — Cluster {cid}', fontsize=13, y=1.01)
    
    for i, (_, row) in enumerate(cluster_flights.iterrows()):
        t_rel = row.t - row.t[0]  # time relative to flight start
        
        # Position (3D)
        for dim, label in enumerate(['x', 'y', 'z']):
            axes[i, 0].plot(t_rel, row.pos[:, dim], label=label)
        axes[i, 0].set_ylabel('Position (m)')
        axes[i, 0].legend(fontsize=7, loc='upper right')
        axes[i, 0].set_title(f'Flight {row.flight_idx}', fontsize=9)
        
        # Speed
        axes[i, 1].plot(t_rel, row.speed)
        axes[i, 1].set_ylabel('Speed (m/s)')
        
        # Acceleration magnitude
        axes[i, 2].plot(t_rel, row.acc_mag)
        axes[i, 2].axhline(ACC_THRESHOLD, color='red', linestyle='--', alpha=0.5)
        axes[i, 2].set_ylabel('|Acc| (m/s²)')
        
        # Per-axis acceleration
        acc = row.acc
        for dim, label in enumerate(['x', 'y', 'z']):
            axes[i, 3].plot(t_rel, acc[:, dim], label=label)
        axes[i, 3].set_ylabel('Acc (m/s²)')
        axes[i, 3].legend(fontsize=7, loc='upper right')
        
        if i == n_flights - 1:
            for j in range(4):
                axes[i, j].set_xlabel('Time (s)')
    
    plt.tight_layout()
    plt.show()

## Known case: Dataset_3_29959_240403 clusters 27 vs 28

Compare acceleration profiles between these two clusters side by side.

In [ ]:
session_name = 'Dataset_3_29959_240403'
df = all_session_flights[session_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, cid in zip(axes, [27, 28]):
    cluster_flights = df[df.cluster == cid]
    for _, row in cluster_flights.iterrows():
        t_norm = np.linspace(0, 1, len(row.acc_mag))
        ax.plot(t_norm, row.acc_mag, alpha=0.4, linewidth=1)
    ax.set_title(f'Cluster {cid} (n={len(cluster_flights)})')
    ax.set_xlabel('Fractional time')
    ax.set_ylabel('|Acceleration| (m/s²)')

plt.suptitle(f'{session_name}: Clusters 27 vs 28', fontsize=13)
plt.tight_layout()
plt.show()

# 3D trajectory comparison
fig = plt.figure(figsize=(12, 5))
for i, cid in enumerate([27, 28]):
    ax = fig.add_subplot(1, 2, i + 1, projection='3d')
    cluster_flights = df[df.cluster == cid]
    for _, row in cluster_flights.iterrows():
        ax.plot(row.pos[:, 0], row.pos[:, 1], row.pos[:, 2], alpha=0.3, linewidth=0.8)
    ax.set_title(f'Cluster {cid} (n={len(cluster_flights)})')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.suptitle(f'{session_name}: 3D trajectories', fontsize=13)
plt.tight_layout()
plt.show()